# Eddy pro to fluxy 

Convert the Eddy Pro text file output to a fluxy netcdf file.

Shows an example of processing and of variable names to use in the netcdf file.

In [ ]:
import pandas as pd
from pathlib import Path
from fluxy.test_utils import data_dir
import xarray as xr
from collections import namedtuple
import numpy as np

In [ ]:

var_template = namedtuple(
    'var_template',
    [
        'name_in_file',
        'name_in_output',
        'unit',
    ]
)

def make_variables(sub: str):
    return [
    var_template(
        name_in_file='timeseries',
        name_in_output='time',
        unit='-'
    ),
    var_template(
        name_in_file=f"{sub}_flux",
        name_in_output="ecflux_measured",
        unit="umol m-2 s-1",
    ),
    var_template(
        name_in_file=f"qc_{sub}_flux",
        name_in_output="qa_flag",
        unit="-",
    ),
    var_template(
        name_in_file=f"rand_err_{sub}_flux",
        name_in_output="stdev_ecflux_observed",
        unit="umol m-2 s-1",
    ),
    var_template(
        name_in_file=f"{sub}_strg2",
        name_in_output="ecflux_observed_storage",
        unit="umol m-2 s-1",

    ),
    var_template(
        name_in_file=f"{sub}_molar_density",
        name_in_output="md_observed",
        unit="mol m-3",
    ),
    var_template(
        name_in_file=f"{sub}_mole_fraction",
        name_in_output="mf_observed",
        unit="umol mol-1",
    ),
    var_template(
        name_in_file=f"{sub}_mixing_ratio",
        name_in_output="mr_observed",
        unit="umol mol-1",
    ),
    # I didn't find this variable in the file 
    # it must be under a different name that I don't know
    #var_template(
    #    name_in_file="friction_velocity",
    #    name_in_output="friction_velocity",
    #    unit="m s-1",
    #),
    var_template(
        name_in_file="wind_speed",
        name_in_output="wind_speed",
        unit="m s-1",
    ),
    var_template(
        name_in_file="wind_dir",
        name_in_output="wind_direction",
        unit="degrees",
    ),
    var_template(
        name_in_file='air_temperature',
        name_in_output='air_temperature',
        unit='K',
    ),
    var_template(
        name_in_file='air_pressure',
        name_in_output='air_pressure',
        unit='Pa',
    ),
    var_template(
        name_in_file='pitch',
        name_in_output='pitch',
        unit='degrees',
    ),

]

In [ ]:


eddy_txt_file = data_dir.parent / 'my_data' / "eddy_pro" / "Eddypro_V2.csv"
df = pd.read_csv(eddy_txt_file)
df['timeseries'] = pd.to_datetime(df['timeseries'], format='%Y/%m/%d %H:%M')



sub = 'co2'


variables = make_variables(sub)

df_this = df[[v.name_in_file for v in variables]].rename(
    columns={v.name_in_file: v.name_in_output for v in variables}
)
ds_measurements = df_this.to_xarray()

ds_measurements

In [ ]:
data_dir_simulations = Path(r"C:\Users\coli\Documents\Data\footprints\scaled")
footprint_file = data_dir_simulations / "footprint_emissions_20250422_185952.nc"
vprm_file = data_dir_simulations / "vprm_on_footprints_20250410_110639.nc"

sub_in_sim = {
    "co2": "CO2",
}

# Molar masses of the substances in kg/mol
molar_masses = {
    "CO2": 44.01e-3,
    "CH4": 16.04e-3,
    "N2O": 44.013e-3,
    "NOx": 46.005e-3,
    "CO": 28.01e-3,
}


def read_icos_cites_flux_simulations(
    file_path: Path | str,
    substance: str,
):
    ds = xr.open_dataset(file_path)

    # Conversion μmol/m2/s = kg/m2/s * μmol/mol / kg/mol
    ds['footprints_temporally_scaled'] = ds['footprints_temporally_scaled'] * 1e6 / molar_masses[substance] 
    ds['footprints_original_scale'] = ds['footprints_original_scale'] * 1e6 / molar_masses[substance] 

    if substance == "CO2":

        ds = ds.sel(substance=["CO2_bio", "CO2_fos"]).sum(dim="substance")

        # Add vprm
        ds_vprm = xr.open_dataset(vprm_file)
        da_vprm = ds_vprm[["GEE", "RE"]].to_dataarray(dim="category")

        ds = xr.Dataset(
            {
                "footprints_temporally_scaled": xr.concat(
                    [ds["footprints_temporally_scaled"], da_vprm], dim="category"
                ),
                "footprints_original_scale": xr.concat(
                    [ds["footprints_original_scale"], da_vprm], dim="category"
                ),
                "footprint_sum": ds["footprint_sum"],
                "blh_qc": ds["blh_qc"],
            }
        )

    else:
        ds = ds.sel(substance=substance)

    ds = ds.rename({"category": "sector"}).rename_vars(
        {
            "footprints_temporally_scaled": "ecflux_sectorial_prior",
            "footprints_original_scale": "ecflux_sectorial_prior_unscaled",
            "footprint_sum": "footprint_coverage_fraction",
            "blh_qc": "qa_blh",
        }
    )

    # Assign the units 
    ds["ecflux_sectorial_prior"].attrs["units"] = "umol m-2 s-1"
    ds["ecflux_sectorial_prior"].attrs["description"] = "prior simulation of sectorial fluxes, scaled with temporal profiles for each categories"
    ds["ecflux_sectorial_prior_unscaled"].attrs["units"] = "umol m-2 s-1"
    ds["ecflux_sectorial_prior"].attrs["description"] = "prior simulation of sectorial fluxes, without scaling with temporal profiles"
    ds.attrs['species'] = substance
    ds[
        "footprint_coverage_fraction"
    ] *= 100  # convert from m2 to fraction (10x10m grid cells)

    return ds


ds_simulated = read_icos_cites_flux_simulations(
    footprint_file, substance=sub_in_sim[sub]
)
ds_simulated

In [ ]:
# Merge the dataset on time
# First, set 'time' as the dimension instead of 'index' for ds
ds_time = ds_measurements.swap_dims({'index': 'time'})

# Merge on the 'time' dimension
ds = xr.merge([ds_time, ds_simulated.rename({'datetime': 'time'})], join="outer", compat="override")
ds = ds.assign(index=('time', np.arange(len(ds.time), dtype=int)))  # Ensure index is set correctly
#ds = ds.set_index({'time': 'index'})  # Set 'time' as the index for consistency
# Swap time and index 
ds = ds.swap_dims({'time': 'index'})  # Swap 'time' and 'index' dimensions
ds = ds.drop_vars([
    'timestep'
])
ds

In [ ]:
# Platform format (there is an index axis which is the main and then platforms (here just one))
# that are refered to with the number_of_identifier variable
ds = ds.assign(
    platform=('platform', ['Hardau']),
    number_of_identifier=('index', np.zeros(len(df_this), dtype=int)),
).set_coords(['time', 'number_of_identifier'])
# Assign units
for v in variables:
    if v.name_in_output == 'time':
        continue
    ds[v.name_in_output].attrs['units'] = v.unit

In [ ]:
import datetime
ds = ds.assign_attrs(
    {
        'title': 'Eddy covariance fluxes from Hardau',
        'institution': 'Empa/University of Basel',
        'source': 'Eddypro V2',
        'comment': 'Data processed with Eddypro V2 for fluxy plots',
        'creation_date': datetime.datetime.now().isoformat(),
        "fluxy_substance": sub,
        "fluxy_data_type": "eddy_flux",
        "fluxy_model": "EDDY_HARDAU",
    }
)
upper_sub = sub.upper()
out_file_path = eddy_txt_file.parent / 'EDDY' / upper_sub / f'EDDY_HARDAU_{upper_sub}_eddy_flux.nc'
out_file_path.parent.mkdir(parents=True, exist_ok=True)
ds.to_netcdf(out_file_path)

## Part 2: An additional storage file which only contains the storage flux variable

In [ ]:
# Another file which contains only the storage term
storage_file = eddy_txt_file.parent / "co2_strg_2layers_5point_20220801-20240731.csv"
df_storage = pd.read_csv(storage_file)
df_storage = df_storage.rename(
    columns={
        'Datetime_UTC_center': 'time',
        'co2_strg_2layers': "ecflux_observed_storage",
    }
)
# The data is the middle of the 30 minutes interval
df_storage['time'] = pd.to_datetime(df_storage['time'], format='%Y-%m-%d %H:%M:%S') - pd.Timedelta(minutes=15)
ds_storage = df_storage.to_xarray()

ds_ontime = ds.swap_dims({'index': 'time'})
# Fill with nan the missing times
ds_storage_on_time = ds_storage.swap_dims({'index': 'time'}).reindex(
    time=ds_ontime.time,)
ds_ontime["ecflux_observed_storage"] = ds_storage_on_time["ecflux_observed_storage"]

    

ds_storage = ds_ontime.swap_dims({'time': 'index'})  # Swap 'time' and 'index' dimensions
# units 
ds_storage["ecflux_observed_storage"].attrs['units'] = "umol m-2 s-1"
ds_storage = ds_storage.assign_attrs(
    {
        'title': 'Eddy covariance storage fluxes from Hardau with 2-layer model',
        'institution': 'Empa/University of Basel',
        'source': 'Eddypro V2',
        'comment': 'Data processed with Eddypro V2 for fluxy plots',
        'creation_date': datetime.datetime.now().isoformat(),
        "fluxy_substance": sub,
        "fluxy_data_type": "eddy_flux",
        "fluxy_model": "EDDY_HARDAU_STORAGE_2LAYERS",
    }
)
out_file_path_storage = eddy_txt_file.parent / 'EDDY' / upper_sub / f'EDDY_HARDAU_STORAGE_2LAYERS_{upper_sub}_eddy_flux.nc'
ds_storage.to_netcdf(out_file_path_storage)